![image.png](https://i.imgur.com/a3uAqnb.png)

# **🐾 Pet Segmentation with U-Net**
In this lab, we will:  
✅ Use the **Oxford-IIIT Pet Dataset** for segmentation  
✅ Build a **custom U-Net model with residual connections**  
✅ Train and evaluate the U-Net model  

## **1️⃣ Dataset Class**

The Oxford-IIIT Pet dataset is preloaded using PyTorch's `OxfordIIITPet` API. We preprocess:
1. **Images**: Normalize and resize  
2. **Targets**: Convert segmentation masks to tensors  

In [ ]:
from torchvision.datasets import OxfordIIITPet
from torchvision import transforms
from torch.utils.data import DataLoader
from torch import nn

# Define transforms for images
img_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

# need to subtract 1 from the make to bring it from 1-3 range to 0-2 range (CrossEntropyLoss requires labels to start from 0)
class SubtractOne(nn.Module):
  def forward(self, img):
    return img-1

# Define transforms for masks
target_transforms = transforms.Compose([                    ## Notice how we have a transform for the target (because it is an image)
    transforms.Resize((256, 256)),                          ##        and another one for the image itself.
    transforms.PILToTensor(),
    SubtractOne()                                           ## Question: What do you think would happen if we
                                                            ##  added rotation augmentation to the image only?
])

# Load train and test datasets
train_dataset = OxfordIIITPet(
    root='data/train/', split="trainval", target_types="segmentation", download=True,
    transform=img_transforms, target_transform=target_transforms
)

test_dataset = OxfordIIITPet(
    root='data/test/', split="test", target_types="segmentation", download=True,
    transform=img_transforms, target_transform=target_transforms
)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train Dataset: {len(train_dataset)} images")
print(f"Test Dataset: {len(test_dataset)} images")

Train Dataset: 3680 images
Test Dataset: 3669 images


In [ ]:
train_dataset

Dataset OxfordIIITPet
    Number of datapoints: 3680
    Root location: data/train/
    StandardTransform
Transform: Compose(
               Resize(size=(256, 256), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
           )
Target transform: Compose(
                      Resize(size=(256, 256), interpolation=bilinear, max_size=None, antialias=True)
                      PILToTensor()
                      SubtractOne()
                  )

### Let's display some images

## **2️⃣ Model Class**

#### We define a **U-Net** model with skip connections (residuals) implemented using **concatenation**.


## **3️⃣ Training and Validation Loops**

To train the U-Net model, we use **Binary Cross Entropy (BCE) Loss**.


## **4️⃣ Running Training**

## **🔹 Why `out_channels=3` in U-Net?**
Our segmentation model needs to **predict probabilities** for each pixel belonging to **one of 3 classes** (background, pet, and boundary).  

✅ Instead of predicting a **single-channel mask**, we predict **3 channels**, where:  
- **Channel 0** → Probability of **background**  
- **Channel 1** → Probability of **pet**  
- **Channel 2** → Probability of **boundary**  

🚀 **How does it work?**
- The output of U-Net is a **(batch_size, 3, height, width)** tensor.
- We use `CrossEntropyLoss`, which expects a **multi-class probability map**.
- The **target masks** should be in **label-encoded format** (`(batch_size, height, width)` with values `0, 1, 2` for each pixel).
- In inference, we could take `argmax(dim=1)`, which picks the class with the highest probability for each pixel.



### Plot loss

## **5️⃣ Visualizing Predictions**

To evaluate the model, we compare the predicted segmentation masks with the ground truth masks.


### Contributed by: Mohamed Eltayeb